In [ ]:
"""
CHAT (.cha) parser + feature extraction pipeline.

Outputs (in output/):
 - utterance_level_full.csv
 - participant_features.csv
 - visit_features.csv
 - sbert_embeddings.npy   
"""

import os
import re
import string
from pathlib import Path
from typing import List, Optional, Tuple
import numpy as np
import pandas as pd

# Optional dependencies
try:
    from sentence_transformers import SentenceTransformer
    SBERT_AVAILABLE = True
except Exception:
    SBERT_AVAILABLE = False

try:
    import pylangacq as pla
    PYLANGACQ_AVAILABLE = True
except Exception:
    PYLANGACQ_AVAILABLE = False

# ---------------- CONFIG ----------------
FOLDER_PATH = "/work/MaleneJensen#0692/exam/data/Transcripts_KL checked"  
SBERT_MODEL = "all-mpnet-base-v2"        
COMPUTE_SBERT = SBERT_AVAILABLE          
BATCH_SIZE = 64                          
# ----------------------------------------

# CHAT markers
CHAT_FILLED_PAUSES = ["&-ah", "&-eh", "&-er", "&-ew", "&-hm", "&-mm", "&-uh", "&-uhm", "&-um"]
CHAT_UNINTELL = ["xxx", "yyy"]  # kept in raw, removed in clean utterances

PRONOUN_SET = {
    "i","you","he","she","they","we","me","him","her","them",
    "my","your","his","hers","our","their","mine","yours"
}

# ---------------- Helper functions ----------------

def parse_filename(filename: str) -> Tuple[Optional[str], Optional[int]]:
    m = re.match(r"(.+?)\.visit(\d+)", filename, flags=re.IGNORECASE)
    if not m:
        return None, None
    return m.group(1), int(m.group(2))

def determine_group(ID: str) -> str:
    """
    Heuristic:
    - If ID matches initials pattern: 2-3 uppercase letters for some followed by one digit -> TD
    - If ID matches Name pattern: One uppercase followed by lowercase letters -> ASD
    - Otherwise default to ASD
    """
    # Initials pattern (TD): 2-3 uppercase letters + possibly a digit
    if re.fullmatch(r"[A-Z]{2,3}\d*", ID):
        return "TD"
    # Name-like pattern (ASD): Capital + lowercase letters (e.g., James)
    if re.fullmatch(r"[A-Z][a-z]+", ID):
        return "ASD"
    # Fallback
    return "ASD"

def extract_header_info(filepath: str) -> Tuple[Optional[str], Optional[str]]:
    gender = None
    age = None
    try:
        with open(filepath, "r", encoding="utf-8") as fh:
            for line in fh:
                if "@Sex of CHI:" in line:
                    gender = line.split(":", 1)[1].strip()
                if "@Age of CHI:" in line:
                    age = line.split(":", 1)[1].strip()
                if gender and age:
                    break
    except Exception:
        # fallback reading
        with open(filepath, "r", encoding="utf-8", errors="ignore") as fh:
            for line in fh:
                if "@Sex of CHI:" in line:
                    gender = line.split(":", 1)[1].strip()
                if "@Age of CHI:" in line:
                    age = line.split(":", 1)[1].strip()
                if gender and age:
                    break
    return gender, age

def extract_turns_from_file(filepath: str) -> List[Tuple[str, str]]:
    turns = []
    try:
        with open(filepath, "r", encoding="utf-8") as fh:
            for line in fh:
                if not line.startswith("*"):
                    continue
                parts = line.split(":", 1)
                if len(parts) < 2:
                    continue
                speaker = parts[0].lstrip("*").strip()
                content = parts[1].strip()
                turns.append((speaker, content))
    except Exception:
        with open(filepath, "r", encoding="utf-8", errors="ignore") as fh:
            for line in fh:
                if not line.startswith("*"):
                    continue
                parts = line.split(":", 1)
                if len(parts) < 2:
                    continue
                speaker = parts[0].lstrip("*").strip()
                content = parts[1].strip()
                turns.append((speaker, content))
    return turns

# ---------------- Cleaning helpers ----------------

def underscore_to_spaces(text: Optional[str]) -> str:
    return "" if text is None else text.replace("_", " ")

def remove_chat_unintelligible(text: str) -> str:
    if not text:
        return text
    # Remove xxx and yyy
    return re.sub(r"\b(?:xxx|yyy)\b", " ", text, flags=re.IGNORECASE)

def remove_chat_filled_pauses(text: str) -> str:
    if not text:
        return text
    pattern = r"\b(?:" + "|".join(re.escape(t) for t in CHAT_FILLED_PAUSES) + r")\b"
    return re.sub(pattern, " ", text, flags=re.IGNORECASE)

def remove_standard_punctuation(text: str) -> str:
    if not text:
        return ""
    return re.sub(r"[.,!?;:\"'()\[\]\{\}<>]", " ", text)

def normalize_whitespace_lower(text: str) -> str:
    if not text:
        return ""
    s = re.sub(r"\s+", " ", text).strip()
    return s.lower()

def _remove_chat_markers(text: str) -> str:
    """
    Cleaning:
      - remove morphological markers: @sl, @o, @si, etc.
      - remove repetitions: [/], [//], [?], [:], [=! ...]
      - remove pauses: (.) (..) (...)
      - split/slash words correctly: / → space
      - remove broken words: ba^by -> baby
      - remove prolongations: :::
      - remove &+fragment
      - keep word content if recoverable
    """
    if not text:
        return ""

    s = text
    # Remove CHAT attribute suffixes: token@something -> token
    s = re.sub(r"\b(\w+)@\w+\b", r"\1", s)
    # Remove repetition markers
    s = re.sub(r"\[/?/?\]", " ", s)   # [/], [//], [///]
    # Remove question / comment markers
    s = re.sub(r"\[\?\]", " ", s)
    s = re.sub(r"\[:\]", " ", s)
    s = re.sub(r"\[=[^]]*\]", " ", s)
    # Remove pauses (.) (..) (...)
    s = re.sub(r"\(\.{1,3}\)", " ", s)
    # Prolongations: go::: → go
    s = re.sub(r"(\w+):{2,}", r"\1", s)
    # Broken words: ba^by → baby
    s = re.sub(r"(\w+)\^(\w*)", r"\1\2", s)
    # Slash words: "more / more" -> "more more"
    s = s.replace("/", " ")
    # Remove &+phonological fragments
    s = re.sub(r"&\+\w*", " ", s)
    # Remove standalone CHAT markers like &: or unclear markers
    s = re.sub(r"&-[a-z]+", " ", s)
    return s

def clean_chat_utterance_for_metrics(raw: Optional[str]) -> str:
    """
    Cleaning:
      - remove CHAT markers but keep lexical content
      - remove filled pauses, unintelligible tokens, punctuation
      - lowercase + normalize
    """
    if raw is None:
        return ""

    s = raw
    # Step 1: underscores → spaces
    s = underscore_to_spaces(s)
    # Step 2: remove CHAT filled pauses (&-uh etc.)
    s = remove_chat_filled_pauses(s)
    # Step 3: remove CHAT unintelligible (xxx, yyy)
    s = remove_chat_unintelligible(s)
    # Step 4: remove all CHAT markers / repairs / codes
    s = _remove_chat_markers(s)
    # Step 5: remove punctuation
    s = remove_standard_punctuation(s)
    # Step 6: turn into single-space lowercase
    s = normalize_whitespace_lower(s)

    return s

# ---------------- Disfluency detection (RAW) ----------------

def count_chat_disfluencies(utt_raw: Optional[str]) -> int:
    if not utt_raw:
        return 0
    s = utt_raw.lower()
    total = 0
    # filled pauses
    for fp in CHAT_FILLED_PAUSES:
        total += len(re.findall(re.escape(fp), s))
    # pauses (.), (..), (...)
    total += len(re.findall(r"\(\.{1,3}\)", s))
    # repetition marker
    total += len(re.findall(re.escape("[/]"), s))
    # revision marker
    total += len(re.findall(re.escape("[//]"), s))
    # prolongations (word:::)
    total += len(re.findall(r"\w+:::+", s))
    # broken words (ba^by)
    total += len(re.findall(r"\w+\^\w*", s))
    # phonological &+ fragment
    total += len(re.findall(r"&\+\w*", s))
    # partial words with &-
    total += len(re.findall(r"&-\w+", s)) - sum(len(re.findall(re.escape(fp), s)) for fp in CHAT_FILLED_PAUSES)
    # unintelligible tokens
    total += len(re.findall(r"\bxxx\b", s))
    total += len(re.findall(r"\byyy\b", s))
    return int(total)

def within_utterance_repeats_count_from_clean(clean: str) -> int:
    if not clean:
        return 0
    return len(re.findall(r"\b(\w+)(?:\s+\1){1,}\b", clean))

def pronoun_count_from_clean(clean: str) -> int:
    if not clean:
        return 0
    return sum(1 for t in clean.split() if t in PRONOUN_SET)

# ---------------- VocD wrapper ----------------

def compute_vocd_for_list(utterances: List[str]) -> Optional[float]:
    if not PYLANGACQ_AVAILABLE:
        return None
    try:
        return pla.vocd(utterances)
    except Exception:
        return None

# ---------------- Main pipeline ----------------

def parse_cha_folder_and_extract(folder_path: str, compute_sbert: bool = COMPUTE_SBERT,
                                 sbert_model_name: str = SBERT_MODEL, batch_size: int = BATCH_SIZE):
    rows = []
    files = sorted([f for f in os.listdir(folder_path) if f.endswith(".cha")])
    for filename in files:
        filepath = os.path.join(folder_path, filename)
        ID, Visit = parse_filename(filename)
        if ID is None:
            continue
        Group = determine_group(ID)
        gender_header, age_header = extract_header_info(filepath)
        turns = extract_turns_from_file(filepath)
        previous_turn_text = None
        previous_turn_speaker = None
        for (speaker, content) in turns:
            if speaker.upper() == "CHI":
                utt_raw = content
                utt_clean = clean_chat_utterance_for_metrics(utt_raw)
                utterance_length = len(utt_clean.split()) if utt_clean.strip() else 0 
                unique_utt = len(set(utt_clean.split())) if utt_clean.strip() else 0
                disfluency_count = count_chat_disfluencies(utt_raw)
                within_reps = within_utterance_repeats_count_from_clean(utt_clean)
                pronoun_count = pronoun_count_from_clean(utt_clean)
                repeats_previous = False
                echo_of_adult = False
                if previous_turn_text is not None:
                    prev_norm = clean_chat_utterance_for_metrics(previous_turn_text)
                    curr_norm = utt_clean
                    if prev_norm and curr_norm and prev_norm == curr_norm:
                        repeats_previous = True
                        if previous_turn_speaker and previous_turn_speaker.upper() != "CHI":
                            echo_of_adult = True
                rows.append({
                    "ID": ID,
                    "Visit": int(Visit),
                    "Group": Group,
                    "Gender": gender_header,
                    "Age": age_header,
                    "Filename": filename,
                    "Path": filepath,
                    "Utterance_raw": utt_raw,
                    "Utterance_clean": utt_clean,
                    "Utterance_length": utterance_length,
                    "UniqueWords_utterance": unique_utt,
                    "Disfluency_count": disfluency_count,
                    "WithinUtterance_Repeats": within_reps,
                    "RepeatsPreviousTurn": repeats_previous,
                    "EchoOfAdult": echo_of_adult,
                    "PronounCount": pronoun_count
                })
            previous_turn_text = content
            previous_turn_speaker = speaker

    df_utts = pd.DataFrame(rows)
    if df_utts.empty:
        print("No CHI utterances extracted. Check folder path and file format.")
        return df_utts, pd.DataFrame(), pd.DataFrame(), None

    # Propagate gender across visits safely
    gender_map = (
        df_utts.groupby("ID")["Gender"]
        .apply(lambda s: s.dropna().iloc[0] if s.dropna().shape[0] > 0 else None)
        .to_dict()
    )
    df_utts["Gender"] = df_utts.apply(lambda r: gender_map.get(r["ID"], r["Gender"]), axis=1)

    # Ensure Visit int
    df_utts["Visit"] = df_utts["Visit"].astype(int)

    # Aggregations: IQR helper
    def iqr(arr):
        if len(arr) == 0:
            return np.nan
        return float(np.percentile(arr, 75) - np.percentile(arr, 25))

    # Visit-level aggregations
    visit_agg = df_utts.groupby(["ID", "Visit"]).agg(
        Utterance_count=("Utterance_clean", "count"),
        MLU_visit=("Utterance_length", "mean"),
        IQR_LU_visit=("Utterance_length", lambda x: iqr(list(x))),
        UniqueWords_visit=("Utterance_clean", lambda uts: len(set(" ".join(uts).split()))),
        Disfluency_count_visit=("Disfluency_count", "sum"),
        RepeatsPrev_prop_visit=("RepeatsPreviousTurn", "mean"),
        EchoOfAdult_prop_visit=("EchoOfAdult", "mean"),
        AvgWithinRepeats_visit=("WithinUtterance_Repeats", "mean"),
        AvgPronounCount_visit=("PronounCount", "mean")
    ).reset_index()

    # Participant-level aggregations
    part_agg = df_utts.groupby("ID").agg(
        Utterance_count_part=("Utterance_clean", "count"),
        MLU_part=("Utterance_length", "mean"),
        IQR_LU_part=("Utterance_length", lambda x: iqr(list(x))),
        UniqueWords_participant=("Utterance_clean", lambda uts: len(set(" ".join(uts).split()))),
        Disfluency_count_part=("Disfluency_count", "sum"),
        RepeatsPrev_prop_part=("RepeatsPreviousTurn", "mean"),
        EchoOfAdult_prop_part=("EchoOfAdult", "mean"),
        AvgWithinRepeats_part=("WithinUtterance_Repeats", "mean"),
        AvgPronounCount_part=("PronounCount", "mean")
    ).reset_index()

    # VocD per participant (if available)
    if PYLANGACQ_AVAILABLE:
        vocd_list = []
        for ID, grp in df_utts.groupby("ID"):
            utterances_clean = grp["Utterance_clean"].tolist()
            vocd_val = compute_vocd_for_list(utterances_clean)
            vocd_list.append((ID, vocd_val))
        df_vocd = pd.DataFrame(vocd_list, columns=["ID", "VocD"])
        part_agg = part_agg.merge(df_vocd, on="ID", how="left")
    else:
        part_agg["VocD"] = None

    # Merge demographics into aggregates
    demo = df_utts.groupby("ID").agg({
        "Gender": lambda s: next((x for x in s if pd.notnull(x)), None),
        "Age": lambda s: next((x for x in s if pd.notnull(x)), None),
        "Group": lambda s: next((x for x in s if pd.notnull(x)), None)
    }).reset_index()
    part_agg = part_agg.merge(demo, on="ID", how="left")
    visit_agg = visit_agg.merge(demo, on="ID", how="left")

    # SBERT embeddings
    sbert_embeddings = None
    if compute_sbert and SBERT_AVAILABLE:
        #print("Computing SBERT embeddings (batch_size={}): this may take time...".format(batch_size))
        model = SentenceTransformer(sbert_model_name)
        texts = df_utts["Utterance_clean"].fillna("").tolist()
        all_embs = []
        for i in range(0, len(texts), batch_size):
            batch = texts[i:i+batch_size]
            emb = model.encode(batch, show_progress_bar=True)
            all_embs.append(emb)
        sbert_embeddings = np.vstack(all_embs)
        df_utts = df_utts.reset_index(drop=True)
        df_utts["SBERT_emb_index"] = df_utts.index
    else:
        df_utts["SBERT_emb_index"] = None
        if compute_sbert and not SBERT_AVAILABLE:
            print("SBERT requested but sentence-transformers not installed; skip embeddings.")

    # Reorder columns and make final utterance-level DF
    rename_map = {
        "Utterance_raw": "raw_utterance",
        "Utterance_clean": "clean_utterance"
    }
    df_utts = df_utts.rename(columns=rename_map)
    # Ensure the requested column order appears
    wanted_cols = ["ID", "Age", "Gender", "Group", "raw_utterance", "clean_utterance"]
    # append the rest in a sensible order
    other_cols = [c for c in df_utts.columns if c not in wanted_cols]
    df_utts = df_utts[wanted_cols + other_cols]

    return df_utts, part_agg, visit_agg, sbert_embeddings

# ---------------- Run pipeline & save ----------------

if __name__ == "__main__":
    folder_path = FOLDER_PATH
    compute_sbert = COMPUTE_SBERT and SBERT_AVAILABLE

    print("Parsing .cha files and extracting features...")
    df_utts, df_part, df_visit, sbert_embs = parse_cha_folder_and_extract(folder_path, compute_sbert=compute_sbert,
                                                                           sbert_model_name=SBERT_MODEL,
                                                                           batch_size=BATCH_SIZE)

    out_dir = Path("output")
    out_dir.mkdir(parents=True, exist_ok=True)

    # Save utterance-level parquet & csv
    utt_parquet = out_dir / "utterance_level_full.parquet"
    df_utts.to_parquet(utt_parquet, index=False)
    print(f"Saved utterance-level dataframe to {utt_parquet} (rows: {len(df_utts)})")

    utt_csv = out_dir / "utterance_level_full.csv"
    df_utts.to_csv(utt_csv, index=False)
    print(f"Saved utterance-level dataframe CSV to {utt_csv}")

    # Save visit-level and participant-level
    visit_path = out_dir / "visit_features.csv"
    df_visit.to_csv(visit_path, index=False)
    print(f"Saved visit-level features to {visit_path}")

    part_path = out_dir / "participant_features.csv"
    df_part.to_csv(part_path, index=False)
    print(f"Saved participant-level features to {part_path}")

    # Save SBERT embeddings if present
    if sbert_embs is not None:
        emb_path = out_dir / "sbert_embeddings.npy"
        np.save(emb_path, sbert_embs)
        print(f"Saved SBERT embeddings to {emb_path}")

    print("Pipeline finished.")
